In [2]:
import os
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# 1. Wczytanie danych z pliku .env
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
API_KEY = os.getenv("FOOTBALL_API_KEY")

# 2. Utworzenie połączenia z bazą MySQL
connection_string = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
engine = create_engine(connection_string)

# 3. Pobranie meczów z API (Premier League - kod 'PL')
url = "https://api.football-data.org/v4/competitions/PL/matches?limit=10"
headers = {"X-Auth-Token": API_KEY}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    matches = response.json().get("matches", [])
    print(f"Pobrano {len(matches)} meczów z API.")

    # 4. Zapytanie SQL zapisujące dane (zapobiega dublowaniu!)
    insert_query = text("""
        INSERT INTO matches (id, competition_name, home_team, away_team, score_home, score_away, status, match_date)
        VALUES (:id, :comp_name, :home, :away, :score_home, :score_away, :status, :match_date)
        ON DUPLICATE KEY UPDATE
            score_home = VALUES(score_home),
            score_away = VALUES(score_away),
            status = VALUES(status);
    """)

    # 5. Wykonanie zapisu w bazie
    with engine.begin() as conn:
        for m in matches:
            conn.execute(insert_query, {
                "id": m["id"],
                "comp_name": m["competition"]["name"],
                "home": m["homeTeam"]["name"],
                "away": m["awayTeam"]["name"],
                "score_home": m["score"]["fullTime"]["home"],
                "score_away": m["score"]["fullTime"]["away"],
                "status": m["status"],
                "match_date": m["utcDate"].replace("T", " ").replace("Z", "")
            })
            
    print("Dane zostały pomyślnie zapisane do MySQL!")
else:
    print(f"Błąd pobierania danych z API. Kod statusu: {response.status_code}")

Pobrano 380 meczów z API.
Dane zostały pomyślnie zapisane do MySQL!


In [7]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Połączenie z bazą
load_dotenv()
engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

# 2. Pobranie wszystkich danych z bazy
df = pd.read_sql("SELECT * FROM matches", con=engine)

# 3. Podstawowe podsumowanie bazy
print("=" * 40)
print("📊 PODSTAWOWE INFORMACJE O BAZIE DANYCH")
print("=" * 40)
print(f"Liczba wszystkich rekordów (meczów): {len(df)}")
print(f"Nazwy kolumn w bazie: {list(df.columns)}")
print(f"Zakres dat meczów: od {df['match_date'].min()} do {df['match_date'].max()}")

print("\n--- Breakdown według statusu meczu ---")
print(df["status"].value_counts())

print("\n--- Statystyki goli ---")
# Filtrujemy tylko rozegrane mecze (gdzie są wyniki)
finished_matches = df[df["status"] == "FINISHED"].copy()

if not finished_matches.empty:
    finished_matches["total_goals"] = (
        finished_matches["score_home"] + finished_matches["score_away"]
    )
    print(f"Rozegrane mecze: {len(finished_matches)}")
    print(
        f"Łączna liczba goli: {int(finished_matches['total_goals'].sum())}"
    )
    print(
        f"Średnia goli na mecz: {finished_matches['total_goals'].mean():.2f}"
    )
else:
    print("Brak rozegranych meczów z wynikami w pobranej próbce.")

# 4. Podgląd pierwszych 5 wierszy z bazy
print("\n--- PODGLĄD 5 PIERWSZYCH REKORDÓW ---")
df.head()

📊 PODSTAWOWE INFORMACJE O BAZIE DANYCH
Liczba wszystkich rekordów (meczów): 380
Nazwy kolumn w bazie: ['id', 'competition_name', 'home_team', 'away_team', 'score_home', 'score_away', 'status', 'match_date', 'updated_at']
Zakres dat meczów: od 2026-08-21 19:00:00 do 2027-05-30 12:00:00

--- Breakdown według statusu meczu ---
status
SCHEDULED    380
Name: count, dtype: int64

--- Statystyki goli ---
Brak rozegranych meczów z wynikami w pobranej próbce.

--- PODGLĄD 5 PIERWSZYCH REKORDÓW ---


,id,competition_name,home_team,away_team,score_home,score_away,status,match_date,updated_at
0,560542,Premier League,Arsenal FC,Coventry City FC,None,None,SCHEDULED,2026-08-21 19:00:00,2026-07-23 00:51:15
1,560543,Premier League,Hull City AFC,Manchester United FC,None,None,SCHEDULED,2026-08-22 11:30:00,2026-07-23 00:51:15
2,560544,Premier League,Ipswich Town FC,Sunderland AFC,None,None,SCHEDULED,2026-08-22 14:00:00,2026-07-23 00:51:15
3,560545,Premier League,Nottingham Forest FC,Leeds United FC,None,None,SCHEDULED,2026-08-22 14:00:00,2026-07-23 00:51:15
4,560546,Premier League,Everton FC,Crystal Palace FC,None,None,SCHEDULED,2026-08-22 14:00:00,2026-07-23 00:51:15
